In [1]:
# Numerical Computing with JAX

## `jax.numpy`: The Familiar Interface

In [2]:
import jax.numpy as jnp
import numpy as np
import jax

# These look identical
x_np = np.array([1.0, 2.0, 3.0])
x_jnp = jnp.array([1.0, 2.0, 3.0])

print("NumPy type:", type(x_np))
print("JAX type:", type(x_jnp))

# Same API
print("sin:", np.sin(x_np))
print("sin:", jnp.sin(x_jnp))
print("norm:", np.linalg.norm(x_np))
print("norm:", jnp.linalg.norm(x_jnp))

NumPy type: <class 'numpy.ndarray'>
JAX type: <class 'jaxlib._jax.ArrayImpl'>
sin: [0.84147098 0.90929743 0.14112001]
sin: [0.84147096 0.9092974  0.14112   ]
norm: 3.7416573867739413
norm: 3.7416575


## Array Immutability

In [3]:
# NumPy: in-place mutation works
x_np = np.array([1.0, 2.0, 3.0])
x_np[0] = 99.0
print("NumPy after mutation:", x_np)

# JAX: in-place mutation raises an error
x_jnp = jnp.array([1.0, 2.0, 3.0])
try:
    x_jnp[0] = 99.0
except TypeError as e:
    print("JAX error:", e)

NumPy after mutation: [99.  2.  3.]
JAX error: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


In [4]:
x = jnp.array([1.0, 2.0, 3.0])
x_new = x.at[0].set(99.0)

print("Original (unchanged):", x)
print("Updated copy:", x_new)

Original (unchanged): [1. 2. 3.]
Updated copy: [99.  2.  3.]


In [5]:
x = jnp.array([1.0, 2.0, 3.0, 4.0])

print("set  :", x.at[1].set(10.0))
print("add  :", x.at[1].add(10.0))
print("mul  :", x.at[1].mul(10.0))
print("min  :", x.at[1].min(0.5))
print("max  :", x.at[1].max(5.0))

set  : [ 1. 10.  3.  4.]
add  : [ 1. 12.  3.  4.]
mul  : [ 1. 20.  3.  4.]
min  : [1.  0.5 3.  4. ]
max  : [1. 5. 3. 4.]


## Slice Assignment and Multi-Index Updates

In [6]:
x = jnp.zeros(8)

# Slice update
print(x.at[2:5].set(1.0))

# Multiple indices
print(x.at[jnp.array([0, 3, 6])].set(jnp.array([10.0, 20.0, 30.0])))

# 2D update
A = jnp.zeros((4, 4))
print(A.at[1, 2].set(5.0))
print(A.at[0, :].set(jnp.ones(4)))

[0. 0. 1. 1. 1. 0. 0. 0.]
[10.  0.  0. 20.  0.  0. 30.  0.]
[[0. 0. 0. 0.]
 [0. 0. 5. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[1. 1. 1. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


## JAX's PRNG: Explicit, Stateless Keys

In [7]:
import jax.random as jrandom

# Create an initial key from a seed
key = jrandom.PRNGKey(42)
print("Key:", key)
print("Key type:", type(key))
print("Key dtype:", key.dtype)

Key: [ 0 42]
Key type: <class 'jaxlib._jax.ArrayImpl'>
Key dtype: uint32


In [8]:
# Generating random values — the key does NOT change
x1 = jrandom.normal(key, shape=(3,))
x2 = jrandom.normal(key, shape=(3,))

print("x1:", x1)
print("x2:", x2)
print("Same?", jnp.all(x1 == x2))

x1: [-0.02830462  0.46713185  0.29570296]
x2: [-0.02830462  0.46713185  0.29570296]
Same? True


## The split Pattern

In [9]:
key = jrandom.PRNGKey(0)

# Split into two subkeys
key, subkey = jrandom.split(key)

a = jrandom.normal(subkey, (3,))
print("a:", a)

# Split again for the next draw
key, subkey = jrandom.split(key)
b = jrandom.normal(subkey, (3,))
print("b:", b)
print("Different?", jnp.any(a != b))

a: [-2.4424558  -2.0356805   0.20554423]
b: [-1.2574776 -0.4016044 -1.1213601]
Different? True


In [10]:
# Split into multiple subkeys at once
key = jrandom.PRNGKey(0)
subkeys = jrandom.split(key, 5)

samples = jnp.stack([jrandom.normal(k, ()) for k in subkeys])
print("5 independent samples:", samples)

5 independent samples: [ 1.0040143 -2.4424558  1.2956359 -0.6219436 -1.0208853]


## Why Stateless PRNG? Reproducibility Under `jit`

In [11]:
import functools


@functools.partial(jax.jit, static_argnums=(1,))
def generate(key, n):
    return jrandom.normal(key, (n,))


key = jrandom.PRNGKey(42)

# Same key → same output, always
r1 = generate(key, 4)
r2 = generate(key, 4)
print("Same key, same output:", jnp.all(r1 == r2))

# Split for different outputs
key1, key2 = jrandom.split(key)
r3 = generate(key1, 4)
r4 = generate(key2, 4)
print("Different keys, different output:", jnp.any(r3 != r4))

Same key, same output: True
Different keys, different output: True


## Default Precision: 32 vs 64 Bit

In [12]:
# Default: float32
print(jnp.array(1.0).dtype)  # float32
print(jnp.array([1, 2, 3]).dtype)  # int32

# NumPy default: float64
print(np.array(1.0).dtype)  # float64

float32
int32
float64


In [13]:
# Enable 64-bit mode globally
jax.config.update("jax_enable_x64", True)

print(jnp.array(1.0).dtype)  # float64 now

float64


In [14]:
# Or use explicit dtype
x32 = jnp.array(1.0, dtype=jnp.float32)
x64 = jnp.array(1.0, dtype=jnp.float64)
print("32-bit:", x32.dtype)
print("64-bit:", x64.dtype)

32-bit: float32
64-bit: float64


In [15]:
# Restore the course default (32-bit) for the remaining sections
jax.config.update("jax_enable_x64", False)
print(jnp.array(1.0).dtype)  # float32 again

float32


## Out-of-Bounds Indexing: A Silent Difference

In [16]:
x = jnp.array([1.0, 2.0, 3.0])

# NumPy: raises IndexError
try:
    print(np.array([1.0, 2.0, 3.0])[10])
except IndexError as e:
    print("NumPy raises:", e)

# JAX: clips to valid range silently
print("JAX clips:", x[10])  # returns x[2] = 3.0
print("JAX clips negative:", x[-10])  # returns x[0] = 1.0

NumPy raises: index 10 is out of bounds for axis 0 with size 3
JAX clips: 3.0
JAX clips negative: 1.0


## Converting Between NumPy and JAX

In [17]:
# NumPy to JAX: copies data to device
x_np = np.array([1.0, 2.0, 3.0])
x_jax = jnp.array(x_np)
print("np → jax:", x_jax)

# JAX to NumPy: copies data to host
x_back = np.array(x_jax)
print("jax → np:", x_back)
print("Roundtrip match:", np.allclose(x_np, x_back))

np → jax: [1. 2. 3.]
jax → np: [1. 2. 3.]
Roundtrip match: True


In [18]:
# jax.device_put to move to a specific device
device = jax.devices()[0]
x_on_device = jax.device_put(x_np, device)
print("Device:", x_on_device.devices())

Device: {CpuDevice(id=0)}


## Summary of Key Differences

In [19]:
# 1. In-place mutation
x_np = np.zeros(4)
x_np[0] = 1.0  # NumPy: works
x_jax = jnp.zeros(4)
x_jax = x_jax.at[0].set(1.0)  # JAX: functional update

# 2. Random numbers
np.random.seed(42)
r_np = np.random.randn(3)

key = jrandom.PRNGKey(42)
key, subkey = jrandom.split(key)
r_jax = jrandom.normal(subkey, (3,))

# 3. Default dtype
print("np default:", np.array(1.0).dtype)  # float64
print("jax default:", jnp.array(1.0).dtype)  # float32

# 4. Out-of-bounds
# np.array([1,2,3])[10]  → IndexError
# jnp.array([1,2,3])[10] → 3.0 (clipped)

print("\nAll differences accounted for — safe to proceed")

np default: float64
jax default: float32

All differences accounted for — safe to proceed
